In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

####Customers

In [0]:
@dp.table(name="silver_customers")
@dp.expect_or_drop(
    "customer_id_not_null",
    "customer_id IS NOT NULL"
)
def silver_customers():

    return (
        spark.read.table(
            "dd_hr_sandpit.ecommerce_project.bronze_customers"
        )
        .dropDuplicates(["customer_id"])
    )

####Products

In [0]:
@dp.table(name="silver_products")
@dp.expect_or_drop(
    "valid_price",
    "price > 0"
)
def silver_products():

    return (
        spark.read.table(
            "dd_hr_sandpit.ecommerce_project.bronze_products"
        )
        .withColumn(
            "price",
            col("price").cast("double")
        )
        .dropDuplicates(["product_id"])
    )

######3) Clickstream

In [0]:
@dp.table(name="silver_clickstream")
@dp.expect_or_drop(
    "valid_event_type",
    "event_type IN ('view','add_to_cart','purchase')"
)
def silver_clickstream():

    return (
        spark.read.table(
            "dd_hr_sandpit.ecommerce_project.bronze_clickstream"
        )
        .withColumn(
            "event_timestamp",
            col("timestamp").cast("timestamp")
        )
        .dropDuplicates(["event_id"])
    )